# 01 - Amazon India Sales Data Cleaning & Preprocessing
**Project:** Amazon India Sales Analytics & AI-Powered Business Intelligence System  
**Role:** Data Science & Analytics Intern Portfolio Project  
**Author:** Manoj (for Executive Manager Ravi & Recruiter Review)

---

### Notebook Objective:
1. **Data Understanding**: Inspect the raw Amazon India sales dataset (schema, shape, types, missing values, duplicates).
2. **Data Filtering & Quality Checks**: Validate numerical ranges, identify anomalies in order statuses, prices, discounts.
3. **Data Preprocessing & Formatting**: Parse order dates, convert data types, and normalize categorical fields.
4. **Feature Enrichment**: Derive business columns:
   - Calendar features: `Year`, `Month`, `Month_Name`, `Year_Month`, `Day`, `Day_of_Week`, `Is_Weekend`, `Quarter`.
   - Financial KPIs: `Profit_Margin_Pct`, `Is_Returned`, `Is_Cancelled`, `Realized_Sales_INR`, `Lost_Sales_INR`.
5. **Data Export**: Save the clean, enriched dataset to `data/processed/amazon_sales_cleaned.csv` for downstream EDA, SQL, and Machine Learning.


In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)


Pandas version: 2.2.3
NumPy version: 2.1.3


## Step 1: Load Raw Dataset
We load the raw Amazon India Sales dataset and inspect its dimensions.


In [ ]:
# Load the dataset
raw_path = "../data/raw/amazon_sales.csv"
if not os.path.exists(raw_path):
    raw_path = "../Amazon Sales Data India.xlsx"

df_raw = pd.read_csv(raw_path) if raw_path.endswith('.csv') else pd.read_excel(raw_path)

print(f"Dataset Shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
df_raw.head()


Dataset Shape: 10,000 rows, 13 columns


,Order_ID,Order_Date,Category,Product,Quantity,Unit_Price_INR,Discount_Pct,Total_Sales_INR,Profit_INR,Payment_Method,Fulfillment,Order_Status,Ship_State
0,IN-AMD-100000,2024-04-12,Apparel & Fashion,Saree with Unstitched Blouse,3,1129.14,0.25,2540.56,702.96,Cash on Delivery (COD),Seller Flex,Delivered,Telangana
1,IN-AMD-100001,2025-03-11,Apparel & Fashion,Men Denim Jeans,3,1159.01,0.05,3303.18,770.96,UPI,Amazon (FBA),Shipped,Kerala
2,IN-AMD-100002,2026-05-10,Beauty & Personal Care,Ubtan Face Wash,1,287.77,0.00,287.77,86.73,UPI,Amazon (FBA),Delivered,West Bengal
3,IN-AMD-100003,2024-09-27,Electronics & Mobiles,Wireless Earbuds,4,20928.14,0.05,79526.93,26466.63,Credit/Debit Card,Amazon (FBA),Shipped,Rajasthan
4,IN-AMD-100004,2024-04-16,Beauty & Personal Care,Sunscreen Gel SPF 50,1,931.65,0.15,791.90,217.90,UPI,Seller Flex,Delivered,Karnataka


## Step 2: Schema Inspection & Missing Value Audit
Checking data types, non-null counts, and identifying potential missing values or anomalies.


In [ ]:
# Column info & data types
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         10000 non-null  object 
 1   Order_Date       10000 non-null  object 
 2   Category         10000 non-null  object 
 3   Product          10000 non-null  object 
 4   Quantity         10000 non-null  int64  
 5   Unit_Price_INR   10000 non-null  float64
 6   Discount_Pct     10000 non-null  float64
 7   Total_Sales_INR  10000 non-null  float64
 8   Profit_INR       10000 non-null  float64
 9   Payment_Method   10000 non-null  object 
 10  Fulfillment      10000 non-null  object 
 11  Order_Status     10000 non-null  object 
 12  Ship_State       10000 non-null  object 
dtypes: float64(4), int64(1), object(8)
memory usage: 1015.8+ KB


In [ ]:
# Missing values count per column
missing_summary = pd.DataFrame({
    'Missing_Count': df_raw.isnull().sum(),
    'Missing_Pct (%)': (df_raw.isnull().sum() / len(df_raw)) * 100
})
missing_summary


,Missing_Count,Missing_Pct (%)
Order_ID,0,0.0
Order_Date,0,0.0
Category,0,0.0
Product,0,0.0
Quantity,0,0.0
Unit_Price_INR,0,0.0
Discount_Pct,0,0.0
Total_Sales_INR,0,0.0
Profit_INR,0,0.0
Payment_Method,0,0.0


## Step 3: Duplicate Record Detection
Ensuring primary key uniqueness and overall row integrity.


In [ ]:
duplicate_rows = df_raw.duplicated().sum()
unique_orders = df_raw['Order_ID'].nunique()

print(f"Total duplicate rows: {duplicate_rows}")
print(f"Unique Order IDs: {unique_orders:,} out of {len(df_raw):,} records")


Total duplicate rows: 0
Unique Order IDs: 10,000 out of 10,000 records


## Step 4: Data Transformation & DateTime Parsing
- Convert `Order_Date` to pandas `datetime64[ns]` format.
- Sort chronologically by `Order_Date` to preserve time-series continuity.
- Verify numerical columns: `Quantity`, `Unit_Price_INR`, `Discount_Pct`, `Total_Sales_INR`, `Profit_INR`.


In [ ]:
df_clean = df_raw.copy()

# Parse datetime
df_clean['Order_Date'] = pd.to_datetime(df_clean['Order_Date'])

# Sort by date
df_clean = df_clean.sort_values('Order_Date').reset_index(drop=True)

# Numerical type casting
numeric_cols = ['Quantity', 'Unit_Price_INR', 'Discount_Pct', 'Total_Sales_INR', 'Profit_INR']
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Categorical stripping
cat_cols = ['Category', 'Product', 'Payment_Method', 'Fulfillment', 'Order_Status', 'Ship_State']
for col in cat_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

print(f"Date Range: from {df_clean['Order_Date'].min().date()} to {df_clean['Order_Date'].max().date()}")
df_clean.dtypes


Date Range: from 2024-01-01 to 2026-08-30


Order_ID                   object
Order_Date         datetime64[ns]
Category                   object
Product                    object
Quantity                    int64
Unit_Price_INR            float64
Discount_Pct              float64
Total_Sales_INR           float64
Profit_INR                float64
Payment_Method             object
Fulfillment                object
Order_Status               object
Ship_State                 object
dtype: object

## Step 5: Feature Enrichment for Business Analytics & ML
We construct essential business metrics directly mapping to dashboard requirements:
- `Year`, `Month`, `Month_Name`, `Year_Month`, `Day`, `Day_of_Week`, `Day_Name`, `Is_Weekend`, `Quarter`
- `Profit_Margin_Pct` = `(Profit_INR / Total_Sales_INR) * 100`
- `Is_Delivered`, `Is_Shipped`, `Is_Returned`, `Is_Cancelled`, `Is_Loss_Order`
- `Realized_Sales_INR` vs `Lost_Sales_INR`


In [ ]:
# 1. Temporal Features
df_clean['Year'] = df_clean['Order_Date'].dt.year
df_clean['Month'] = df_clean['Order_Date'].dt.month
df_clean['Month_Name'] = df_clean['Order_Date'].dt.strftime('%b')
df_clean['Year_Month'] = df_clean['Order_Date'].dt.strftime('%Y-%m')
df_clean['Day'] = df_clean['Order_Date'].dt.day
df_clean['Day_of_Week'] = df_clean['Order_Date'].dt.dayofweek
df_clean['Day_Name'] = df_clean['Order_Date'].dt.strftime('%A')
df_clean['Is_Weekend'] = df_clean['Day_of_Week'].isin([5, 6]).astype(int)
df_clean['Quarter'] = df_clean['Order_Date'].dt.quarter

# 2. Financial KPIs
df_clean['Profit_Margin_Pct'] = np.where(
    df_clean['Total_Sales_INR'] > 0,
    np.round((df_clean['Profit_INR'] / df_clean['Total_Sales_INR']) * 100, 2),
    0.0
)

# 3. Order Status Flags
df_clean['Is_Delivered'] = (df_clean['Order_Status'] == 'Delivered').astype(int)
df_clean['Is_Shipped'] = (df_clean['Order_Status'] == 'Shipped').astype(int)
df_clean['Is_Returned'] = (df_clean['Order_Status'] == 'Returned').astype(int)
df_clean['Is_Cancelled'] = (df_clean['Order_Status'] == 'Cancelled').astype(int)
df_clean['Is_Loss_Order'] = df_clean['Order_Status'].isin(['Returned', 'Cancelled']).astype(int)

# 4. Realized vs Lost Sales
df_clean['Realized_Sales_INR'] = np.where(
    df_clean['Order_Status'].isin(['Returned', 'Cancelled']),
    0.0,
    df_clean['Total_Sales_INR']
)
df_clean['Lost_Sales_INR'] = np.where(
    df_clean['Order_Status'].isin(['Returned', 'Cancelled']),
    df_clean['Total_Sales_INR'],
    0.0
)

df_clean[['Order_ID', 'Order_Date', 'Category', 'Total_Sales_INR', 'Profit_INR', 'Profit_Margin_Pct', 'Order_Status', 'Lost_Sales_INR']].head()


,Order_ID,Order_Date,Category,Total_Sales_INR,Profit_INR,Profit_Margin_Pct,Order_Status,Lost_Sales_INR
0,IN-AMD-104794,2024-01-01,Apparel & Fashion,1817.90,626.25,34.45,Delivered,0.00
1,IN-AMD-106959,2024-01-01,Apparel & Fashion,1712.38,389.15,22.73,Delivered,0.00
2,IN-AMD-101026,2024-01-01,Home & Kitchen,5790.49,754.78,13.03,Delivered,0.00
3,IN-AMD-101327,2024-01-01,Home & Kitchen,30916.76,10215.79,33.04,Delivered,0.00
4,IN-AMD-102682,2024-01-01,Apparel & Fashion,2268.09,0.00,0.00,Returned,2268.09


## Step 6: Data Sanity Checks & Statistical Overview
Inspect the summary statistics of the numeric features to verify data integrity.


In [ ]:
summary_stats = df_clean[['Quantity', 'Unit_Price_INR', 'Discount_Pct', 'Total_Sales_INR', 'Profit_INR', 'Profit_Margin_Pct']].describe()
summary_stats.round(2)


,Quantity,Unit_Price_INR,Discount_Pct,Total_Sales_INR,Profit_INR,Profit_Margin_Pct
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,2.49,6821.40,0.09,15578.99,3316.70,21.25
std,1.11,9183.50,0.08,24159.35,5805.86,9.46
min,1.00,150.27,0.00,118.82,0.00,0.00
25%,2.00,1002.82,0.00,1931.24,299.33,15.90
50%,2.00,2290.08,0.10,4795.80,891.62,22.32
75%,3.00,7628.62,0.15,17357.33,3250.53,28.79
max,4.00,34992.80,0.25,139647.92,45596.30,35.00


## Step 7: Export Processed Dataset
Save the cleaned, validated dataset to `data/processed/amazon_sales_cleaned.csv`.


In [ ]:
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "amazon_sales_cleaned.csv")

df_clean.to_csv(output_file, index=False)
print(f"Processed dataset successfully exported to: {output_file}")
print(f"Total Records: {len(df_clean):,}")
print(f"Total Columns: {df_clean.shape[1]}")


Processed dataset successfully exported to: ../data/processed\amazon_sales_cleaned.csv
Total Records: 10,000
Total Columns: 30
